# 08 — Primitive Sets and Dropping Sets

**Motivation:** A *primitive set* is a set A ⊂ Z>1 where no member divides another.
Erdős proved f(A) = Σ 1/(a log a) is bounded; Lichtman (2022) proved primes maximize it.

**Question:** How do Collatz dropping sets relate to primitive sets?
- Are dropping sets (or their subgroups) close to primitive?
- What does f(A) look like for dropping-set subgroups vs primes?
- Does the divisibility structure within/between sets reveal new patterns?

In [ ]:
import sys
sys.path.insert(0, '..')

import math
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from itertools import combinations

from collatz import core, dropping, factorization
from collatz.utils import members_of_class

LIMIT = 10_000  # upper bound for enumeration
print(f'Exploration limit: n < {LIMIT}')

## 1. Divisibility Structure Within Dropping Sets

In [ ]:
# Classify all integers up to LIMIT by dropping set
drop_classes = defaultdict(list)
for n in range(2, LIMIT):
    k = dropping.dropping_set(n)
    drop_classes[k].append(n)

print(f'Dropping sets found: {sorted(drop_classes.keys())}')
print(f'Sizes: {{{k}: {len(v)} for k, v in sorted(drop_classes.items())[:15]}}}')

In [ ]:
def count_divisor_pairs(members):
    """Count (a, b) pairs in members where a | b, a != b."""
    members_set = set(members)
    count = 0
    for a in members:
        # Check multiples of a that are also in the set
        multiple = 2 * a
        while multiple < LIMIT:
            if multiple in members_set:
                count += 1
            multiple += a
    return count

def maximal_primitive_subset(members):
    """Greedy extraction: take elements in order, skip if divisible by any kept element."""
    kept = []
    for a in sorted(members):
        if not any(a % k == 0 for k in kept):
            kept.append(a)
    return kept

# Analyze each dropping set
results = []
for k in sorted(drop_classes.keys()):
    members = drop_classes[k]
    n_members = len(members)
    if n_members < 2:
        continue
    
    div_pairs = count_divisor_pairs(members)
    total_pairs = n_members * (n_members - 1) // 2
    prim_subset = maximal_primitive_subset(members)
    
    results.append({
        'set_k': k,
        'size': n_members,
        'divisor_pairs': div_pairs,
        'total_pairs': total_pairs,
        'div_pair_ratio': div_pairs / total_pairs if total_pairs > 0 else 0,
        'primitive_subset_size': len(prim_subset),
        'primitive_fraction': len(prim_subset) / n_members,
    })

df_div = pd.DataFrame(results)
print('=== Divisibility Structure Within Each Dropping Set ===')
print(df_div.to_string(index=False, float_format='%.6f'))

In [ ]:
# Examine Set_1 (all evens) vs Set_3 vs higher sets
print('=== Detailed look at divisor structure ===')
for k in sorted(drop_classes.keys())[:8]:
    members = drop_classes[k]
    prim = maximal_primitive_subset(members)
    print(f'\nSet_{k}: {len(members)} members, {len(prim)} in max primitive subset ({len(prim)/len(members):.1%})')
    print(f'  First 20 members: {members[:20]}')
    print(f'  First 20 primitive: {prim[:20]}')
    # What gets excluded?
    excluded = [m for m in members[:30] if m not in set(prim)]
    if excluded:
        print(f'  Excluded (first few): {excluded[:10]} — each divisible by a smaller member')

## 2. Cross-Set Divisibility: Does $n \in \text{Set}_k$ and $d | n$ constrain $d$'s set?

In [ ]:
# For each divisor pair (a, b) where a | b, what are their respective dropping sets?
cross_set_pairs = Counter()  # (set_of_a, set_of_b) -> count

# Focus on odd numbers (more interesting — evens always in Set_1)
odd_members = {k: [m for m in v if m % 2 == 1] for k, v in drop_classes.items()}

# Build a lookup: n -> set_k for odd n
odd_class = {}
for k, members in odd_members.items():
    for m in members:
        odd_class[m] = k

# Find all odd divisor pairs
for a in range(3, LIMIT, 2):
    if a not in odd_class:
        continue
    k_a = odd_class[a]
    multiple = 3 * a  # smallest odd multiple
    while multiple < LIMIT:
        if multiple in odd_class:
            k_b = odd_class[multiple]
            cross_set_pairs[(k_a, k_b)] += 1
        multiple += 2 * a  # next odd multiple

print('=== Cross-Set Divisibility (odd numbers only) ===')
print('(set_of_divisor, set_of_multiple) -> count')
for (ka, kb), cnt in sorted(cross_set_pairs.items(), key=lambda x: -x[1])[:20]:
    print(f'  Set_{ka} -> Set_{kb}: {cnt} pairs')

In [ ]:
# Heatmap: does divisibility prefer certain set transitions?
all_ks = sorted(set(k for (k, _) in cross_set_pairs) | set(k for (_, k) in cross_set_pairs))
heat = pd.DataFrame(0, index=all_ks[:12], columns=all_ks[:12], dtype=float)

for (ka, kb), cnt in cross_set_pairs.items():
    if ka in heat.index and kb in heat.columns:
        heat.loc[ka, kb] = cnt

# Normalize rows to show conditional distribution: given divisor in Set_k, where are multiples?
heat_norm = heat.div(heat.sum(axis=1), axis=0).fillna(0)

print('=== Conditional distribution: P(multiple in Set_j | divisor in Set_i) ===')
print('Rows = divisor set, Cols = multiple set')
print(heat_norm.to_string(float_format='%.3f'))

## 3. The Erdős f-sum: f(A) = Σ 1/(a log a)

In [ ]:
def f_sum(members):
    """Compute Erdős f-sum = Σ 1/(a * log(a)) for members > 1."""
    return sum(1.0 / (a * math.log(a)) for a in members if a > 1)

def f_sum_primitive(members):
    """f-sum restricted to the maximal primitive subset."""
    prim = maximal_primitive_subset(members)
    return f_sum(prim), len(prim)

# f(primes) up to LIMIT
primes = [n for n in range(2, LIMIT) if factorization.is_prime(n)]
f_primes = f_sum(primes)
print(f'f(primes) up to {LIMIT}: {f_primes:.6f}  ({len(primes)} primes)')
print(f'Known limit f(P) -> ~1.6366 as LIMIT -> inf')
print()

# f-sum for each dropping set (raw, not primitive)
f_results = []
for k in sorted(drop_classes.keys()):
    members = drop_classes[k]
    if len(members) < 2:
        continue
    f_raw = f_sum(members)
    f_prim, prim_size = f_sum_primitive(members)
    f_results.append({
        'set_k': k,
        'size': len(members),
        'f_raw': f_raw,
        'f_primitive': f_prim,
        'prim_size': prim_size,
        'f_prim / f_primes': f_prim / f_primes,
    })

df_f = pd.DataFrame(f_results)
print('=== Erdős f-sum by Dropping Set ===')
print(df_f.to_string(index=False, float_format='%.6f'))

In [ ]:
# Now by subgroup: (set_k, modulus_m)
subgroup_classes = defaultdict(list)
for n in range(2, LIMIT):
    k = dropping.dropping_set(n)
    m = dropping.dropping_modulus(n)
    subgroup_classes[(k, m)].append(n)

sg_results = []
for (k, m) in sorted(subgroup_classes.keys()):
    members = subgroup_classes[(k, m)]
    if len(members) < 3:
        continue
    f_raw = f_sum(members)
    f_prim, prim_size = f_sum_primitive(members)
    div_pairs = count_divisor_pairs(members)
    
    sg_results.append({
        'set_k': k,
        'mod_m': m,
        'size': len(members),
        'f_raw': f_raw,
        'f_primitive': f_prim,
        'div_pairs': div_pairs,
        'is_primitive': div_pairs == 0,
    })

df_sg = pd.DataFrame(sg_results)
print(f'=== Subgroup Analysis ({len(sg_results)} subgroups) ===')
print(f'Subgroups that ARE primitive (0 divisor pairs): {df_sg["is_primitive"].sum()} / {len(df_sg)}')
print()
print('Top 15 subgroups by f_primitive:')
print(df_sg.nlargest(15, 'f_primitive').to_string(index=False, float_format='%.6f'))

In [ ]:
# Key question: which subgroups are naturally primitive?
primitive_sgs = df_sg[df_sg['is_primitive']]
non_primitive_sgs = df_sg[~df_sg['is_primitive']]

print(f'=== Naturally Primitive Subgroups ===')
if len(primitive_sgs) > 0:
    print(primitive_sgs.to_string(index=False, float_format='%.6f'))
else:
    print('None found — all subgroups contain divisor pairs')

print(f'\n=== Subgroups with fewest divisor pairs (most "nearly primitive") ===')
print(non_primitive_sgs.nsmallest(10, 'div_pairs').to_string(index=False, float_format='%.6f'))

In [ ]:
# Compare: union of all primitive subsets from all dropping sets vs primes
all_primitive_members = []
for k in sorted(drop_classes.keys()):
    members = drop_classes[k]
    prim = maximal_primitive_subset(members)
    all_primitive_members.extend(prim)

# This union isn't itself primitive (cross-set divisibility), so extract primitive from it too
all_primitive_members.sort()
union_prim = maximal_primitive_subset(all_primitive_members)

f_union = f_sum(union_prim)
print(f'Union of per-set primitive subsets: {len(all_primitive_members)} elements')
print(f'After cross-set filtering: {len(union_prim)} primitive elements')
print(f'f(union_primitive) = {f_union:.6f}')
print(f'f(primes)          = {f_primes:.6f}')
print(f'Ratio              = {f_union / f_primes:.6f}')
print()

# How much overlap with primes?
prime_set = set(primes)
union_set = set(union_prim)
overlap = prime_set & union_set
print(f'Primes in union_primitive: {len(overlap)} / {len(primes)} ({len(overlap)/len(primes):.1%})')
print(f'Non-primes in union_primitive: {len(union_set - prime_set)}')
# What are the non-prime elements?
non_prime_in_union = sorted(union_set - prime_set)[:30]
print(f'First non-prime primitive elements: {non_prime_in_union}')

## 4. Primes Within Each Dropping Set

In [ ]:
# How do primes distribute across dropping sets?
prime_by_set = defaultdict(list)
for p in primes:
    k = dropping.dropping_set(p)
    prime_by_set[k].append(p)

print('=== Prime Distribution Across Dropping Sets ===')
prime_dist = []
for k in sorted(prime_by_set.keys()):
    ps = prime_by_set[k]
    f_val = f_sum(ps)
    prime_dist.append({
        'set_k': k,
        'n_primes': len(ps),
        'fraction_of_primes': len(ps) / len(primes),
        'f_sum': f_val,
        'fraction_of_f_primes': f_val / f_primes,
        'first_few': str(ps[:5]),
    })

df_prime_dist = pd.DataFrame(prime_dist)
print(df_prime_dist.to_string(index=False, float_format='%.4f'))

In [ ]:
# Within each dropping set, primes form a primitive subset (trivially — primes are always primitive)
# Compare f(primes_in_Set_k) vs f(primitive_subset_of_Set_k)
print('=== Primes vs Max Primitive Subset per Dropping Set ===')
comparison = []
for k in sorted(drop_classes.keys()):
    members = drop_classes[k]
    if len(members) < 5:
        continue
    primes_in_k = [m for m in members if factorization.is_prime(m)]
    prim = maximal_primitive_subset(members)
    
    f_pk = f_sum(primes_in_k) if primes_in_k else 0
    f_prim = f_sum(prim)
    
    comparison.append({
        'set_k': k,
        'n_primes': len(primes_in_k),
        'n_primitive': len(prim),
        'f_primes_in_k': f_pk,
        'f_primitive_k': f_prim,
        'primes_dominate': f_pk >= f_prim,
    })

df_comp = pd.DataFrame(comparison)
print(df_comp.to_string(index=False, float_format='%.6f'))
print(f'\nPrimes dominate in {df_comp["primes_dominate"].sum()} / {len(df_comp)} sets')

## 5. Key Structural Question: Does Dropping Set Membership Respect Divisibility?

In [ ]:
# If a | b (both odd), is there a pattern in set(b) - set(a)?
# The affine structure tells us dest(n) = slope * n + C within subgroups.
# Does divisibility interact with this?

set_diffs = []
for a in range(3, 500, 2):  # odd divisors
    k_a = odd_class.get(a)
    if k_a is None:
        continue
    for mult in range(3, LIMIT // a + 1, 2):  # odd multipliers
        b = a * mult
        if b >= LIMIT:
            break
        k_b = odd_class.get(b)
        if k_b is None:
            continue
        set_diffs.append({
            'a': a, 'b': b, 'mult': mult,
            'set_a': k_a, 'set_b': k_b,
            'set_diff': k_b - k_a,
            'mult_is_prime': factorization.is_prime(mult),
        })

df_diffs = pd.DataFrame(set_diffs)
print(f'Total odd divisor pairs: {len(df_diffs)}')
print(f'\n=== Distribution of set_b - set_a ===')
print(df_diffs['set_diff'].describe())
print(f'\nMost common set differences:')
print(df_diffs['set_diff'].value_counts().head(15))

In [ ]:
# Does the multiplier determine the set shift?
# Group by multiplier value and see if set_diff is consistent
print('=== Set shift by multiplier (first 20 odd multipliers) ===')
for mult in range(3, 43, 2):
    sub = df_diffs[df_diffs['mult'] == mult]
    if len(sub) < 5:
        continue
    diffs = sub['set_diff'].value_counts()
    entropy = -(diffs / diffs.sum() * np.log2(diffs / diffs.sum())).sum()
    print(f'  mult={mult:3d} (prime={factorization.is_prime(mult)}): '
          f'{len(sub)} pairs, {len(diffs)} distinct diffs, '
          f'entropy={entropy:.2f} bits, top: {dict(diffs.head(4))}')

## 6. Findings

### Finding 1: Higher Dropping Set Subgroups Are Naturally Primitive
**43 out of 52 subgroups (83%) are primitive sets** — zero divisor pairs.

The mechanism: each subgroup (k, m) is an arithmetic progression with period P = 2^(k-s).
When the smallest element exceeds P, consecutive ratios are < 2, so no element can divide another.
Only the "base" subgroups (modulus 0) of Sets 1, 3, 6, 8, 11, 13, 21, 29 have divisor pairs.

### Finding 2: Cross-Set Divisibility is Nearly Uniform
P(multiple in Set_j | divisor in Set_i) is approximately **independent of i**.
This is because ×3 (and all odd multipliers coprime to 2) are bijections on Z/2^B,
so they permute the residue classes that define dropping sets.

### Finding 3: The ×3 Symmetry Theorem 

**Theorem.** For odd n, define shift(n) = set(3n) - set(n). Then:
$$P(\text{shift} = +s) = P(\text{shift} = -s) \quad \text{for all } s > 0$$

**Proof sketch:**
1. n ≡ 1 mod 4 ⟹ n ∈ Set₃, and 3n ≡ 3 mod 4 ⟹ shift always positive
2. n ≡ 3 mod 4 ⟹ n ∈ Set_{k>3}, and 3n ≡ 1 mod 4 ⟹ 3n ∈ Set₃ ⟹ shift always negative  
3. Since gcd(3, 2^B) = 1, multiplication by 3 is a bijection on Z/2^B
4. So ×3 preserves the distribution of residue classes defining each dropping set
5. P(shift=+s) = density(Set_{3+s}) = P(shift=-s)

### Finding 4: Lattice Path Counting Formula

The density of Set_k among n ≡ 3 mod 4 is **exactly**:

$$\text{density}(\text{Set}_k) = \frac{N(s)}{2^{k-s-2}}$$

where $s$ = Syracuse steps (so $k = \lceil s \cdot \log_2 6 \rceil$) and $N(s)$ counts
**valid alpha prefixes**: sequences $(a_1, \ldots, a_{s-1})$ with:
- $a_1 = 1$ (forced: $3n+1 \equiv 2 \pmod{4}$ for $n \equiv 3 \pmod{4}$)
- $a_i \geq 1$ for all $i$
- $\sum_{j=1}^{i} a_j < i \cdot \log_2 3$ for all $i$ (non-dropping constraint)

The sequence $N(s) = 1, 2, 3, 7, 12, 30, 85, 173, 476, 961, 2652, 8045, \ldots$

Verified: all 7 computable cases match exactly.

| Set_k | shift | s | N(s) | density | decimal |
|-------|-------|---|------|---------|---------|
| 6 | +3 | 2 | 1 | 1/4 | 0.250000 |
| 8 | +5 | 3 | 2 | 1/4 | 0.250000 |
| 11 | +8 | 4 | 3 | 3/32 | 0.093750 |
| 13 | +10 | 5 | 7 | 7/64 | 0.109375 |
| 16 | +13 | 6 | 12 | 3/64 | 0.046875 |
| 19 | +16 | 7 | 30 | 15/512 | 0.029297 |
| 21 | +18 | 8 | 85 | 85/2048 | 0.041504 |

### Finding 5: The Mod-4 Wall
- **Set₃ = exactly {odd n : n ≡ 1 mod 4}**
- All higher sets (6, 8, 11, ...) live in **n ≡ 3 mod 4**
- ×3 always crosses this wall (flips mod-4 class)
- This is why P(shift=0 under ×3) = 0 exactly
- Multipliers p ≡ 1 mod 4 preserve the wall (P(same set) > 50%)
- Multipliers p ≡ 3 mod 4 cross the wall (P(same set) = 0%)

In [ ]:
# WHY does mult by p=5,13,17... preserve set class but mult by 3,7,11... scatter?
# Hypothesis: it's about p mod 4 (or p mod 8)
print('=== Multiplier mod 4 vs set-preserving behavior ===')
for mult in range(3, 43, 2):
    from collections import Counter
    diffs = []
    for a in range(3, LIMIT // mult + 1, 2):
        b = a * mult
        if b < LIMIT:
            ka = odd_class.get(a)
            kb = odd_class.get(b)
            if ka is not None and kb is not None:
                diffs.append(kb - ka)
    if len(diffs) < 10:
        continue
    c = Counter(diffs)
    zero_frac = c.get(0, 0) / len(diffs)
    print(f'  mult={mult:3d}  mod4={mult%4}  mod8={mult%8}  '
          f'P(shift=0)={zero_frac:.3f}  '
          f'prime={factorization.is_prime(mult)}  '
          f'v2(mult-1)={bin(mult-1).count("0") - len(bin(mult-1).lstrip("0"))}')

# Actually compute v2 properly
import math
def v2(n):
    if n == 0: return float('inf')
    v = 0
    while n % 2 == 0:
        v += 1
        n //= 2
    return v

print()
print('=== Refined: v2(mult^2-1) as predictor ===')
for mult in range(3, 43, 2):
    diffs = []
    for a in range(3, LIMIT // mult + 1, 2):
        b = a * mult
        if b < LIMIT:
            ka = odd_class.get(a)
            kb = odd_class.get(b)
            if ka is not None and kb is not None:
                diffs.append(kb - ka)
    if len(diffs) < 10:
        continue
    c = Counter(diffs)
    zero_frac = c.get(0, 0) / len(diffs)
    # mult^2 - 1 = (mult-1)(mult+1), and v2 of this determines how "transparent"
    # the multiplier is to the 2-adic classification
    v2_m2_minus_1 = v2(mult*mult - 1)
    v2_m_minus_1 = v2(mult - 1)
    v2_m_plus_1 = v2(mult + 1)
    print(f'  mult={mult:3d}  v2(m-1)={v2_m_minus_1}  v2(m+1)={v2_m_plus_1}  '
          f'v2(m^2-1)={v2_m2_minus_1}  P(shift=0)={zero_frac:.3f}')